In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Generate Dataset $\mathcal{D} = (X, y)$

In [2]:
import torch

def create_identity_data(n):
    # n = total number of samples
    assert n % 2 == 0, "n must be even"

    half = n // 2

    # Negative and positive values
    X = torch.arange(-half, half, dtype=torch.float32).reshape(-1, 1)

    # Identity relationship: y = x
    y = X.clone()

    return X, y




In [3]:
X, y = create_identity_data(100)



In [4]:
from torch.utils.data import TensorDataset, DataLoader

# Create Dataset
dataset = TensorDataset(X, y)

# Create DataLoader
train_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

print("Dataset size:", len(dataset))

Dataset size: 100


___

# Perceptron

In [5]:
import torch
import torch.nn as nn

class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()

        self.linear = nn.Linear(1, 1)
        self.activation = nn.Identity()

    def forward(self, x):
        x = self.linear(x)
        x = self.activation(x)

        return x


model = Perceptron()

print(model)

Perceptron(
  (linear): Linear(in_features=1, out_features=1, bias=True)
  (activation): Identity()
)


In [6]:
from torch.utils.data import TensorDataset, DataLoader

# Train-test split
train_size = int(0.8 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

# Create datasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 80
Testing samples: 20


In [7]:
criterion = nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.0001
)

epochs = 5000

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # Forward pass
        y_pred = model(X_batch)

        # Loss
        loss = criterion(y_pred, y_batch)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [100/5000], Loss: 0.0128
Epoch [200/5000], Loss: 0.0033
Epoch [300/5000], Loss: 0.0009
Epoch [400/5000], Loss: 0.0002
Epoch [500/5000], Loss: 0.0001
Epoch [600/5000], Loss: 0.0000
Epoch [700/5000], Loss: 0.0000
Epoch [800/5000], Loss: 0.0000
Epoch [900/5000], Loss: 0.0000
Epoch [1000/5000], Loss: 0.0000
Epoch [1100/5000], Loss: 0.0000
Epoch [1200/5000], Loss: 0.0000
Epoch [1300/5000], Loss: 0.0000
Epoch [1400/5000], Loss: 0.0000
Epoch [1500/5000], Loss: 0.0000
Epoch [1600/5000], Loss: 0.0000
Epoch [1700/5000], Loss: 0.0000
Epoch [1800/5000], Loss: 0.0000
Epoch [1900/5000], Loss: 0.0000
Epoch [2000/5000], Loss: 0.0000
Epoch [2100/5000], Loss: 0.0000
Epoch [2200/5000], Loss: 0.0000
Epoch [2300/5000], Loss: 0.0000
Epoch [2400/5000], Loss: 0.0000
Epoch [2500/5000], Loss: 0.0000
Epoch [2600/5000], Loss: 0.0000
Epoch [2700/5000], Loss: 0.0000
Epoch [2800/5000], Loss: 0.0000
Epoch [2900/5000], Loss: 0.0000
Epoch [3000/5000], Loss: 0.0000
Epoch [3100/5000], Loss: 0.0000
Epoch [3200/5000]

In [8]:
import torch

# Random input
x = torch.tensor([[7.0]])

# Evaluation mode
model.eval()

with torch.no_grad():
    prediction = model(x)

print("Input:", x.item())
print("Predicted:", prediction.item())
print("Actual:", x.item())

Input: 7.0
Predicted: 7.0
Actual: 7.0


In [9]:
import torch
import torch.nn as nn

model.eval()

total_squared_error = 0.0
total_absolute_error = 0.0
total_samples = 0

all_predictions = []
all_targets = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        # Prediction
        y_pred = model(X_batch)

        # Store predictions and targets
        all_predictions.append(y_pred)
        all_targets.append(y_batch)

        # Errors
        total_squared_error += torch.sum((y_pred - y_batch) ** 2).item()
        total_absolute_error += torch.sum(torch.abs(y_pred - y_batch)).item()

        total_samples += y_batch.size(0)


# Combine all batches
predictions = torch.cat(all_predictions)
targets = torch.cat(all_targets)

# Metrics
mse = total_squared_error / total_samples
mae = total_absolute_error / total_samples
rmse = mse ** 0.5

# R² score
ss_res = torch.sum((targets - predictions) ** 2)
ss_tot = torch.sum((targets - targets.mean()) ** 2)

r2 = 1 - (ss_res / ss_tot)

print(f"MSE  : {mse:.6f}")
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2.item():.6f}")

MSE  : 0.000000
MAE  : 0.000000
RMSE : 0.000000
R²   : 1.000000


In [10]:
tolerance = 0.1

correct = torch.abs(predictions - targets) <= tolerance

accuracy = correct.float().mean()

print(f"Accuracy (±{tolerance}): {accuracy.item() * 100:.2f}%")

Accuracy (±0.1): 100.00%
